# PDF Toán → MinerU → Word → RAG
### Phiên bản chạy **local trên VSCode** (không dùng Google Colab)

**Luồng xử lý:**

```
PDF toán (local)
   │
   ▼
 MinerU (CLI)  → Markdown + JSON + Ảnh
   │
   ├──► Pandoc  → DOCX (Equation OMML, Heading, Table, Ảnh)  ← "PDF toán -> Word"
   │
   └──► Chunk theo heading → Embedding (multilingual-e5) → FAISS Index → Retriever
                                                                              │
                                                                              ▼
                                                                 Chuẩn bị cho RAG
```

## 1. Kiểm tra môi trường & cài thư viện cần thiết

Cell dưới đây tự động:
1. Kiểm tra phiên bản Python (MinerU 2.x yêu cầu Python 3.10–3.13; trên **Windows**, do
   phụ thuộc `ray`, chỉ 3.10–3.12 được hỗ trợ).
2. Kiểm tra từng thư viện Python cần thiết — nếu thiếu sẽ **tự cài**, nếu có sẽ **kiểm tra
   phiên bản** để tránh xung đột đã biết (numpy/faiss/huggingface_hub/transformers...).
3. Kiểm tra chương trình ngoài `pandoc` (không cài được bằng pip — cần cài theo hệ điều hành).
4. Kiểm tra GPU (CUDA) để chọn cấu hình MinerU/embedding phù hợp.

> Chạy cell này **trước tiên**, mỗi khi mở lại notebook trong VSCode ở máy mới.


In [15]:
import sys
import shutil
import subprocess
import importlib
import importlib.util
import importlib.metadata as ilmd
import platform

print("Python:", sys.version)
print("OS:", platform.system(), platform.release())

PY_MAJOR, PY_MINOR = sys.version_info[:2]
if (PY_MAJOR, PY_MINOR) < (3, 10) or (PY_MAJOR, PY_MINOR) > (3, 13):
    print(f"[CẢNH BÁO] Python {PY_MAJOR}.{PY_MINOR} nằm ngoài khoảng MinerU hỗ trợ chính thức (3.10-3.13).")
if platform.system() == "Windows" and (PY_MAJOR, PY_MINOR) >= (3, 13):
    print("[CẢNH BÁO] Trên Windows, thư viện 'ray' (phụ thuộc của MinerU) không hỗ trợ Python 3.13+. "
          "Khuyến nghị dùng Python 3.10-3.12 (tạo venv/conda riêng).")


def pip_install(*pkgs, upgrade=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        cmd.append("-U")
    cmd.extend(pkgs)
    print("Đang chạy:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("  [LỖI] Cài đặt thất bại:")
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        return False
    return True


def get_installed_version(dist_name):
    try:
        return ilmd.version(dist_name)
    except ilmd.PackageNotFoundError:
        return None


# (import_name, pip_spec, dist_name_for_version_check)
# pip_spec để trống nếu cần cài riêng (torch xử lý bên dưới).
REQUIRED_PACKAGES = [
    ("fitz",                 "pymupdf>=1.24.0",                    "pymupdf"),
    ("numpy",                "numpy>=1.26,<2.1",                   "numpy"),
    ("PIL",                  "pillow>=10.0.0",                     "pillow"),
    ("tqdm",                 "tqdm",                                "tqdm"),
    ("docx",                 "python-docx>=1.1.0",                 "python-docx"),
    ("huggingface_hub",      "huggingface-hub>=0.23.0,<1.0.0",     "huggingface-hub"),
    ("sentence_transformers","sentence-transformers>=3.0.0",       "sentence-transformers"),
    ("faiss",                "faiss-cpu>=1.8.0",                   "faiss-cpu"),
    ("mineru",               'mineru[all]>=2.0,<3',                "mineru"),
]

print("\n=== Kiểm tra thư viện Python ===")
missing = []
for import_name, pip_spec, dist_name in REQUIRED_PACKAGES:
    spec = importlib.util.find_spec(import_name)
    version = get_installed_version(dist_name)
    if spec is None:
        print(f"[THIẾU] {dist_name} -> sẽ cài: {pip_spec}")
        missing.append(pip_spec)
    else:
        print(f"[OK]    {dist_name:<24} version={version}")

if missing:
    print("\nĐang cài các thư viện còn thiếu...")
    pip_install(*missing)
else:
    print("\nTất cả thư viện Python cần thiết đã có sẵn.")

# ---- Torch: cài riêng vì cần chọn build CPU/CUDA phù hợp máy ----
print("\n=== Kiểm tra torch ===")
if importlib.util.find_spec("torch") is None:
    print("[THIẾU] torch -> đang cài bản CPU mặc định (mineru[all] cũng có thể tự kéo torch).")
    print("   Nếu máy có GPU NVIDIA, sau khi cài xong hãy cài lại torch đúng bản CUDA tại:")
    print("   https://pytorch.org/get-started/locally/")
    pip_install("torch")
else:
    import torch
    print("[OK] torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# Nạp lại metadata sau khi có thể vừa cài thêm
importlib.invalidate_caches()

# ---- Kiểm tra xung đột phiên bản đã biết ----
print("\n=== Kiểm tra tương thích phiên bản đã biết ===")
np_ver = get_installed_version("numpy")
faiss_ver = get_installed_version("faiss-cpu")
if np_ver and np_ver.split(".")[0] not in ("1",):
    print(f"[CẢNH BÁO] numpy {np_ver} là numpy 2.x — một số bản build cũ của faiss-cpu / "
          f"pymupdf có thể chưa tương thích. Nếu gặp lỗi 'numpy.dtype size changed', "
          f"chạy: pip install \"numpy<2.1\" --force-reinstall")
hf_ver = get_installed_version("huggingface-hub")
st_ver = get_installed_version("sentence-transformers")
if hf_ver and st_ver:
    print(f"huggingface-hub={hf_ver} | sentence-transformers={st_ver} (yêu cầu hf-hub>=0.23,<1.0 — đã ràng buộc ở trên)")

print("\n=== Kiểm tra Pandoc ===")
pandoc_path = shutil.which("pandoc")
if pandoc_path is None:
    print("[THIẾU] Không tìm thấy 'pandoc' trong PATH.")
    system = platform.system()
    if system == "Windows":
        print("  Cài bằng winget:  winget install --id JohnMacFarlane.Pandoc")
        print("  hoặc choco:       choco install pandoc")
    elif system == "Darwin":
        print("  Cài bằng Homebrew: brew install pandoc")
    else:
        print("  Debian/Ubuntu:  sudo apt-get update && sudo apt-get install -y pandoc")
        print("  Fedora:         sudo dnf install pandoc")
    print("  Hoặc tải trình cài đặt tại: https://github.com/jgm/pandoc/releases")
    print("  Sau khi cài, khởi động lại VSCode / terminal rồi chạy lại cell này.")
else:
    result = subprocess.run(["pandoc", "--version"], capture_output=True, text=True)
    print("[OK] pandoc:", result.stdout.splitlines()[0])

# ---- MinerU CLI ----
print("\n=== Kiểm tra MinerU CLI ===")
mineru_path = shutil.which("mineru")
if mineru_path is None:
    print("[THIẾU] Không tìm thấy lệnh 'mineru' trong PATH (có thể do vừa cài, cần mở lại "
          "terminal/kernel để PATH được nạp lại).")
else:
    result = subprocess.run(["mineru", "--version"], capture_output=True, text=True)
    out = (result.stdout or result.stderr).strip()
    print("[OK] mineru:", out.splitlines()[-1] if out else mineru_path)

print("\n=== Hoàn tất kiểm tra môi trường ===")


Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
OS: Windows 11
[CẢNH BÁO] Python 3.14 nằm ngoài khoảng MinerU hỗ trợ chính thức (3.10-3.13).
[CẢNH BÁO] Trên Windows, thư viện 'ray' (phụ thuộc của MinerU) không hỗ trợ Python 3.13+. Khuyến nghị dùng Python 3.10-3.12 (tạo venv/conda riêng).

=== Kiểm tra thư viện Python ===
[OK]    pymupdf                  version=1.28.2
[OK]    numpy                    version=2.4.2
[OK]    pillow                   version=12.3.0
[OK]    tqdm                     version=4.70.1
[OK]    python-docx              version=1.2.0
[OK]    huggingface-hub          version=1.32.0
[OK]    sentence-transformers    version=6.1.0
[OK]    faiss-cpu                version=1.15.1
[OK]    mineru                   version=4.0.2

Tất cả thư viện Python cần thiết đã có sẵn.

=== Kiểm tra torch ===
[OK] torch 2.10.0+cpu | CUDA available: False

=== Kiểm tra tương thích phiên bản đã biết ===
[CẢNH BÁO] numpy 2.4.2 là numpy 2.x — một số 

## 2. Import thư viện

In [16]:
import os
import re
import json
import shutil
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Optional

import torch
import pymupdf as fitz
from tqdm.auto import tqdm

print("Torch device khả dụng:", "cuda" if torch.cuda.is_available() else "cpu")


Torch device khả dụng: cpu


## 3. Cấu hình thư mục làm việc & nhập file PDF

In [17]:
# ĐIỀN ĐƯỜNG DẪN PDF Ở ĐÂY (nếu muốn) — để trống "" nếu muốn dùng hộp thoại/nhập tay
PDF_INPUT_PATH = ""


def pick_pdf_file(preset_path: str = "") -> Path:
    """Trả về Path tới file PDF hợp lệ, ưu tiên: preset_path -> tkinter dialog -> input()."""
    # 1. Đường dẫn điền sẵn
    if preset_path:
        p = Path(preset_path).expanduser().resolve()
        if p.exists() and p.suffix.lower() == ".pdf":
            return p
        print(f"[CẢNH BÁO] PDF_INPUT_PATH='{preset_path}' không hợp lệ hoặc không tồn tại. "
              f"Sẽ hỏi lại đường dẫn.")

    # 2. Hộp thoại chọn file (chỉ hoạt động khi có màn hình / GUI)
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title="Chọn file PDF toán",
            filetypes=[("PDF files", "*.pdf")]
        )
        root.destroy()
        if selected:
            return Path(selected).expanduser().resolve()
    except Exception:
        pass  

    # 3. Nhập tay qua input()
    while True:
        typed = input("Nhập đường dẫn đầy đủ tới file PDF: ").strip().strip('"')
        p = Path(typed).expanduser().resolve()
        if p.exists() and p.suffix.lower() == ".pdf":
            return p
        print(f"Không tìm thấy PDF hợp lệ tại: {p}. Vui lòng thử lại.")


# ---- Thư mục dự án (tương đương pdf_to_word/ trong bản gốc) ----
WORKSPACE_DIR = Path.cwd().resolve()
PROJECT_DIR = WORKSPACE_DIR / "pdf_to_word"

DIR = {
    "root": PROJECT_DIR,
    "input": PROJECT_DIR / "input",
    "mineru": PROJECT_DIR / "mineru_output",
    "docx": PROJECT_DIR / "docx_output",
    "rag": PROJECT_DIR / "rag_output",
}
for path in DIR.values():
    path.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORKSPACE_DIR)
print("Project  :", PROJECT_DIR)

# ---- Chọn & copy PDF vào thư mục input ----
SELECTED_PDF = pick_pdf_file(PDF_INPUT_PATH)
PROJECT_PDF_PATH = DIR["input"] / SELECTED_PDF.name
shutil.copy2(SELECTED_PDF, PROJECT_PDF_PATH)

print("\nĐã chọn PDF:", SELECTED_PDF)
print("Đã copy vào:", PROJECT_PDF_PATH)


Workspace: C:\Users\ASUS\Documents\CCNLTHT
Project  : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word

Đã chọn PDF: C:\Users\ASUS\Documents\CCNLTHT\LSTM.pdf
Đã copy vào: C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\input\LSTM.pdf


## 4. Kiểm tra PDF (digital / scanned / mixed)

Thông tin này giúp kiểm tra thông tin văn bản từ các trang để biết được trạng thái trang tổng thể của văn bản pdf.
- "digital" nếu toàn bộ văn bản có text_layer.
- "scanned" nếu toàn bộ văn bản không có text layer.
- "mixed" nếu văn bản vừa có các trang có text layer và không có text layer.

In [18]:
pdf_doc = fitz.open(str(PROJECT_PDF_PATH))
NUM_PAGES = len(pdf_doc)

text_lengths = [len(pdf_doc[i].get_text("text")) for i in range(NUM_PAGES)]
pages_with_text = sum(x > 20 for x in text_lengths)
pages_without_text = NUM_PAGES - pages_with_text

if pages_with_text == NUM_PAGES:
    PDF_TYPE = "digital"
elif pages_without_text == NUM_PAGES:
    PDF_TYPE = "scanned"
else:
    PDF_TYPE = "mixed"

pdf_doc.close()

print(f"Số trang            : {NUM_PAGES}")
print(f"Trang có text layer  : {pages_with_text}")
print(f"Trang không có text  : {pages_without_text}")
print(f"Loại PDF             : {PDF_TYPE}")


Số trang            : 46
Trang có text layer  : 46
Trang không có text  : 0
Loại PDF             : digital


## 5. Chạy MinerU

Dùng đúng CLI `mineru`. Backend `pipeline` chạy tốt trên cả CPU và GPU; nếu có GPU, MinerU sẽ tự
tận dụng CUDA để tăng tốc OCR + nhận diện công thức.


In [19]:
MINERU_OUTPUT = DIR["mineru"] / PROJECT_PDF_PATH.stem
MINERU_OUTPUT.mkdir(parents=True, exist_ok=True)

backend = "pipeline"
current_env = os.environ.copy()
current_env["MINERU_BACKEND"] = backend

cmd = [
    "mineru-kit",
    "parse", str(PROJECT_PDF_PATH),
    "-o", str(MINERU_OUTPUT),
]

print("Input :", PROJECT_PDF_PATH)
print("Output:", MINERU_OUTPUT)
print("Chạy  :", " ".join(cmd))
print("\n(Lần chạy đầu tiên MinerU sẽ tự tải model nhận diện layout/công thức, có thể mất vài phút)\n")

result = subprocess.run(
    cmd,
    env=current_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(result.stdout[-4000:])

if result.returncode != 0:
    raise RuntimeError(f"MinerU chạy thất bại (return code {result.returncode}).")

print("\nMinerU hoàn tất.")


Input : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\input\LSTM.pdf
Output: C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM
Chạy  : mineru-kit parse C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\input\LSTM.pdf -o C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM

(Lần chạy đầu tiên MinerU sẽ tự tải model nhận diện layout/công thức, có thể mất vài phút)

/s]

MFR Predict: 100%|██████████| 218/218 [01:55<00:00,  1.88it/s]

OCR-det: 100%|██████████| 428/428 [00:14<00:00, 30.09it/s]
Parsed 1 input(s).


MinerU hoàn tất.


## 6. Tìm Markdown / JSON / ảnh trong output của MinerU

In [20]:
all_files = [p for p in MINERU_OUTPUT.rglob("*") if p.is_file()]
md_files = [p for p in all_files if p.suffix.lower() == ".md"]
json_files = [p for p in all_files if p.suffix.lower() == ".json"]
image_files = [p for p in all_files if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")]

print("Markdown:", len(md_files))
for p in md_files:
    print(" -", p)

print("\nJSON:", len(json_files))
for p in json_files[:20]:
    print(" -", p)

print("\nẢnh:", len(image_files))
for p in image_files[:10]:
    print(" -", p)

if not md_files:
    raise FileNotFoundError("Không tìm thấy Markdown từ MinerU. Kiểm tra log ở cell trước.")

# Chọn Markdown lớn nhất (tránh chọn nhầm file phụ nếu có nhiều .md)
MD_PATH = max(md_files, key=lambda p: p.stat().st_size)
print("\nMarkdown được chọn:", MD_PATH)


Markdown: 1
 - C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM\LSTM.md

JSON: 0

Ảnh: 0

Markdown được chọn: C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM\LSTM.md


## 7. Xem trước Markdown & kiểm tra dấu hiệu công thức LaTeX

In [21]:
markdown_text = MD_PATH.read_text(encoding="utf-8", errors="ignore")

print(markdown_text[:5000])

latex_patterns = [
    r"\\\(", r"\\\[", r"\\begin\{equation", r"\\frac", r"\\int", r"\\sum",
    r"\$\$", r"\$[^\n]+\$",
]

print("\nDấu hiệu LaTeX tìm thấy trong Markdown:")
for pattern in latex_patterns:
    matches = re.findall(pattern, markdown_text)
    print(f"  {pattern:<20} => {len(matches)}")


**Long Short-Term Memory**

**Sepp Hochreiter**

Fakultat f ¨ ur Informatik, Technische Universit ¨ at M¨ unchen, 80290 M ¨ unchen, Germany ¨

Jürgen Schmidhuber

IDSIA, Corso Elvezia 36, 6900 Lugano, Switzerland

**Learning to store information over extended time intervals by recurrent backpropagation takes a very long time, mostly because of insufficient, decaying error backflow. We briefly review Hochreiter’s (1991) analysis of this problem, then address it by introducing a novel, efficient, gradientbased method called long short-term memory (LSTM). Truncating the gradient where this does not do harm, LSTM can learn to bridge minimal time lags in excess of 1000 discrete-time steps by enforcing constant error flow through constant error carousels within special units. Multiplicative gate units learn to open and close access to the constant error flow. LSTM is local in space and time; its computational complexity per time step and weight is** O(1)**. Our experiments with artificial da

## 8. Chuẩn hoá đường dẫn ảnh trong Markdown cho Pandoc

MinerU tham chiếu ảnh bằng đường dẫn tương đối. Pandoc cần đường dẫn tuyệt đối (hoặc
`--resource-path` đúng) để nhúng được ảnh vào DOCX. Công thức LaTeX được **giữ nguyên**,
không bị chuyển thành text thường ở bước này.


In [22]:
image_refs = re.findall(r"!\[[^\]]*\]\(([^)]+)\)", markdown_text)
print("Số lượng tham chiếu ảnh trong Markdown:", len(image_refs))

WORK_MD = DIR["docx"] / f"{PROJECT_PDF_PATH.stem}_for_docx.md"
work_text = markdown_text

for ref in sorted(set(image_refs), key=len, reverse=True):
    clean_ref = ref.strip().strip("<>")
    if clean_ref.startswith(("http://", "https://")):
        continue
    candidate = (MD_PATH.parent / clean_ref).resolve()
    if candidate.exists():
        work_text = work_text.replace(ref, str(candidate.as_posix()))

WORK_MD.write_text(work_text, encoding="utf-8")
print("Đã tạo Markdown để sử dụng:", WORK_MD)


Số lượng tham chiếu ảnh trong Markdown: 4
Đã tạo Markdown để sử dụng: C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\docx_output\LSTM_for_docx.md


## 9. Markdown → DOCX bằng Pandoc

In [23]:
DOCX_PATH = DIR["docx"] / f"{PROJECT_PDF_PATH.stem}.docx"

cmd = [
    "pandoc",
    str(WORK_MD),
    "-o", str(DOCX_PATH),
    "--from=markdown+tex_math_dollars+tex_math_single_backslash",
    "--resource-path", str(MD_PATH.parent),
    "--standalone",
    "--toc",
]

print("Chạy Pandoc...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pandoc tạo DOCX thất bại. Kiểm tra đã cài pandoc (cell mục 1) chưa.")

assert DOCX_PATH.exists() and DOCX_PATH.stat().st_size > 0, "DOCX chưa được tạo hoặc bị rỗng."

print("Đã tạo DOCX     :", DOCX_PATH)
print("Kích thước file :", DOCX_PATH.stat().st_size, "bytes")
print("\n=> Mở file này bằng Microsoft Word / LibreOffice Writer để kiểm tra công thức, "
      "bảng và ảnh.")


Chạy Pandoc...

Đã tạo DOCX     : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\docx_output\LSTM.docx
Kích thước file : 603632 bytes

=> Mở file này bằng Microsoft Word / LibreOffice Writer để kiểm tra công thức, bảng và ảnh.


## 10. Chuẩn bị dữ liệu cho RAG

RAG dùng trực tiếp **Markdown của MinerU** (không đọc ngược từ DOCX), vì Markdown giữ được
LaTeX, heading, cấu trúc — phù hợp hơn để chunk & embedding.


In [24]:
def split_markdown_sections(md_text: str) -> List[Dict[str, str]]:
    sections = []
    current_heading = "Document"
    current_lines: List[str] = []

    for line in md_text.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            if current_lines:
                body = "\n".join(current_lines).strip()
                if body:
                    sections.append({"heading": current_heading, "text": body})
                current_lines = []
            current_heading = re.sub(r"^#{1,6}\s+", "", line).strip()
        else:
            current_lines.append(line)

    if current_lines:
        body = "\n".join(current_lines).strip()
        if body:
            sections.append({"heading": current_heading, "text": body})

    return sections


def make_chunks(sections: List[Dict[str, str]], max_chars: int = 1800, overlap: int = 250) -> List[Dict[str, Any]]:
    chunks = []
    for sec_id, sec in enumerate(sections):
        heading, body = sec["heading"], sec["text"]

        if len(body) <= max_chars:
            chunks.append({
                "chunk_id": len(chunks), "section_id": sec_id,
                "heading": heading, "text": f"# {heading}\n{body}".strip(),
            })
            continue

        start = 0
        while start < len(body):
            end = min(start + max_chars, len(body))
            piece = body[start:end]
            chunks.append({
                "chunk_id": len(chunks), "section_id": sec_id,
                "heading": heading, "text": f"# {heading}\n{piece}".strip(),
            })
            if end >= len(body):
                break
            start = max(0, end - overlap)

    return chunks


sections = split_markdown_sections(markdown_text)
chunks = make_chunks(sections)

print("Số section:", len(sections))
print("Số chunk  :", len(chunks))
if chunks:
    print("\nVí dụ chunk đầu tiên:\n" + chunks[0]["text"][:1000])


Số section: 17
Số chunk  : 349

Ví dụ chunk đầu tiên:
# Document
**Long Short-Term Memory**

**Sepp Hochreiter**

Fakultat f ¨ ur Informatik, Technische Universit ¨ at M¨ unchen, 80290 M ¨ unchen, Germany ¨

Jürgen Schmidhuber

IDSIA, Corso Elvezia 36, 6900 Lugano, Switzerland

**Learning to store information over extended time intervals by recurrent backpropagation takes a very long time, mostly because of insufficient, decaying error backflow. We briefly review Hochreiter’s (1991) analysis of this problem, then address it by introducing a novel, efficient, gradientbased method called long short-term memory (LSTM). Truncating the gradient where this does not do harm, LSTM can learn to bridge minimal time lags in excess of 1000 discrete-time steps by enforcing constant error flow through constant error carousels within special units. Multiplicative gate units learn to open and close access to the constant error flow. LSTM is local in space and time; its computational complexity per tim

## 11. Embedding (sentence-transformers)

In [25]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "intfloat/multilingual-e5-base"
print(f"Đang tải model embedding: {EMBED_MODEL} (lần đầu sẽ tải từ HuggingFace, cần internet)...")
embedder = SentenceTransformer(EMBED_MODEL)

passages = ["passage: " + c["text"] for c in chunks]
embeddings = embedder.encode(passages, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)


Đang tải model embedding: intfloat/multilingual-e5-base (lần đầu sẽ tải từ HuggingFace, cần internet)...


Batches: 100%|██████████| 11/11 [02:21<00:00, 12.84s/it]

Embedding shape: (349, 768)


## 12. Xây FAISS Index

In [26]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

INDEX_PATH = DIR["rag"] / "index.faiss"
CHUNKS_PATH = DIR["rag"] / "chunks.json"

faiss.write_index(index, str(INDEX_PATH))
CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

print("Đã lưu FAISS index :", INDEX_PATH)
print("Đã lưu chunks       :", CHUNKS_PATH)


Đã lưu FAISS index : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\index.faiss
Đã lưu chunks       : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\chunks.json


## 13. Hàm truy hồi (retriever)

In [27]:
def retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    query_embedding = embedder.encode(["query: " + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        item = dict(chunks[int(idx)])
        item["score"] = float(score)
        results.append(item)
    return results


# --- Thử nghiệm nhanh ---
test_question = "Hãy giải thích một khái niệm hoặc công thức trong tài liệu."
results = retrieve(test_question, top_k=5)

for i, r in enumerate(results, 1):
    print(f"--- {i} | score={r['score']:.4f} | {r['heading']} ---")
    print(r["text"][:600])
    print()


--- 1 | score=0.8238 | 4 The Concept of Long Short-Term Memory ---
# 4 The Concept of Long Short-Term Memory
/0Yp8G//DYaT/8AI9H/AA7q/wCCfX/Rinwb/wDDYaT/API9AHx3/wARZ/8AwRV/6Ll4q/8ADfaj/wDGqP8AiLP/AOCKv/RcvFX/AIb7Uf8A41X2J/w7q/4J9f8ARinwb/8ADYaT/wDI9H/Dur/gn1/0Yp8G/wDw2Gk//I9AHx3/AMRZ/wDwRV/6Ll4q/wDDfaj/APGqP+Is/wD4Iq/9Fy8Vf+G+1H/41X2J/wAO6v8Agn1/0Yp8G/8Aw2Gk/wDyPR/w7q/4J9f9GKfBv/w2Gk//ACPQB8d/8RZ//BFX/ouXir/w32o//GqP+Is//gir/wBFy8Vf+G+1H/41X2J/w7q/4J9f9GKfBv8A8NhpP/yPR/w7q/4J9f8ARinwb/8ADYaT/wDI9AHx3/xFn/8ABFX/AKLl4q/8N9qP/wAao/4iz/8Agir/ANFy8Vf+G+1H/wCNV9if8O6v+CfX/Rinwb/8NhpP/wAj0f8ADur/AIJ9f9GKfBv/AMNhpP8A8j0AfHf/ABFn/wDBFX/ouXir/wAN9qP/AMao/wCIs/8A4Iq/9Fy

--- 2 | score=0.8234 | 3.1.3 Intuitive Explanation of Equation 3.2. If ---
# 3.1.3 Intuitive Explanation of Equation 3.2. If
 ] _ { i V } = w _ { i V }$ , u’s incoming weight vector $W _ { u ^ { T } }$ is defined by $[ W _ { u ^ { T } } ] _ { i } : = [ W ] _ { u i } = w _ { u i } ,$ and for $m = 1 , \ldots , q , F ( t - m )$ is t

## 15. Tổng kết pipeline & kiểm tra output cuối cùng

In [28]:
print("===== KẾT QUẢ CUỐI CÙNG =====")
print("PDF gốc      :", PROJECT_PDF_PATH)
print("Markdown     :", MD_PATH)
print("DOCX         :", DOCX_PATH)
print("FAISS index  :", INDEX_PATH)
print("Chunks JSON  :", CHUNKS_PATH)

print("\nKiểm tra tồn tại:")
for p in [PROJECT_PDF_PATH, MD_PATH, DOCX_PATH, INDEX_PATH, CHUNKS_PATH]:
    status = "OK" if Path(p).exists() else "THIẾU"
    print(f"  [{status}] {p}")


===== KẾT QUẢ CUỐI CÙNG =====
PDF gốc      : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\input\LSTM.pdf
Markdown     : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM\LSTM.md
DOCX         : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\docx_output\LSTM.docx
FAISS index  : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\index.faiss
Chunks JSON  : C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\chunks.json

Kiểm tra tồn tại:
  [OK] C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\input\LSTM.pdf
  [OK] C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\mineru_output\LSTM\LSTM.md
  [OK] C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\docx_output\LSTM.docx
  [OK] C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\index.faiss
  [OK] C:\Users\ASUS\Documents\CCNLTHT\pdf_to_word\rag_output\chunks.json
